In [1]:
import os

os.environ['USER'] = 'root'
os.system('pip install ../input/xlearn/xlearn/xlearn-0.40a1/')

import xlearn as xl

In [2]:
import pandas as pd
import xlearn as xl
import numpy as np
import warnings
import optuna
import gc
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from optuna.samplers import TPESampler
from itertools import combinations
from tqdm import tqdm

tqdm.pandas(desc="Progress", leave=True, position=0, ncols=80)
warnings.filterwarnings("ignore", category=FutureWarning)

Using TensorFlow backend.


In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv', index_col='id')
orig = pd.read_csv('/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:

train['Churn'] = train['Churn'].map({'No': 0, 'Yes': 1})
orig['Churn'] = orig['Churn'].map({'No': 0, 'Yes': 1})


In [5]:
_tc = pd.to_numeric(orig.TotalCharges, errors='coerce')
_tc[_tc.isnull()] = orig[_tc.isnull()].MonthlyCharges * orig[_tc.isnull()].tenure
orig.TotalCharges = _tc

In [6]:
CATS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]
print(f"There are {len(CATS)} categorical columns:")
print(CATS)

NUMS = ['tenure', 'MonthlyCharges', 'TotalCharges']
print(f"There are {len(NUMS)} numerical columns:")
print(NUMS)

TARGET = 'Churn'

There are 15 categorical columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
There are 3 numerical columns:
['tenure', 'MonthlyCharges', 'TotalCharges']


In [7]:

DIGITS = []
for c in ['tenure']:
    for k in range(-1, 0):
        n = f'{c}_d{k}'
        train[n] = ((train[c] * 10**k) % 10).fillna(-1).astype("int8")
        test[n] = ((test[c] * 10**k) % 10).fillna(-1).astype("int8")
        orig[n] = ((orig[c] * 10**k) % 10).fillna(-1).astype("int8")
        DIGITS.append(n)

for c in ['MonthlyCharges']:
    for k in range(-2, 2):
        n = f'{c}_d{k}'
        train[n] = ((train[c] * 10**k) % 10).fillna(-1).astype("int8")
        test[n] = ((test[c] * 10**k) % 10).fillna(-1).astype("int8")
        orig[n] = ((orig[c] * 10**k) % 10).fillna(-1).astype("int8")
        DIGITS.append(n)

TE_columns = []
selected_cols =  [
    'Contract',
    'InternetService',
    'PaymentMethod',
    'StreamingTV',
    'PaperlessBilling',
    'TechSupport',
    'StreamingMovies'
]

columns = NUMS + CATS

for r in [2]:
    for cols in tqdm(list(combinations(columns, r))):
        name = '-'.join(cols)
        if name not in selected_cols:
            continue

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        orig[name] = orig[cols[0]].astype(str)
        for col in cols[1:]:
            orig[name] = orig[name] + '_' + orig[col].astype(str)

        combined = pd.concat([train[name], test[name], orig[name]], ignore_index=True)
        combined, _ = combined.factorize()
        if pd.Series(combined).nunique() > len(combined) // 2:
            train = train.drop(name, axis=1)
            test = test.drop(name, axis=1)
            orig = orig.drop(name, axis=1)
            continue
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        orig[name] = combined[len(train) + len(test):]

        TE_columns.append(name)

combined = pd.concat([train, test])
for c in NUMS + CATS:
    combined[c], _ = combined[c].factorize()
train = combined[:len(train)]
test = combined[len(train):]

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

100%|██████████| 153/153 [00:00<00:00, 565898.16it/s]


In [ ]:
class TargetEncoder_(BaseEstimator, TransformerMixin):
    """
    Bộ mã hóa mục tiêu hỗ trợ nhiều hàm tổng hợp,
    kiểm tra chéo nội bộ để ngăn ngừa rò rỉ và làm mịn.
    Tham số
    ---------
    cols_to_encode : danh sách chuỗi
    Danh sách tên cột cần được mã hóa mục tiêu.
    aggs : danh sách chuỗi, mặc định=['mean']
    Danh sách các hàm tổng hợp cần áp dụng. Bất kỳ hàm nào được chấp nhận bởi
    phương thức `.agg()` của pandas đều được hỗ trợ, chẳng hạn như:
    'mean', 'std', 'var', 'min', 'max', 'skew', 'nunique',
    'count', 'sum', 'median'.
    Làm mịn chỉ được áp dụng cho hàm tổng hợp 'mean'.
    cv : số nguyên, mặc định=5
    Số lần fold cho kiểm tra chéo trong fit_transform.
    smooth : số thực hoặc 'auto', mặc định='auto'
    Tham số làm mịn `m`. Giá trị lớn hơn sẽ đặt trọng số lớn hơn cho giá trị trung bình toàn cục. Nếu là 'auto', Phương pháp ước lượng Bayes thực nghiệm được sử dụng.
    drop_original : bool, mặc định=False
    Nếu True, các cột gốc cần được mã hóa sẽ bị loại bỏ.
    """
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=False):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        """
        Học các ánh xạ từ toàn bộ tập dữ liệu.
        Các ánh xạ này được sử dụng cho phương thức chuyển đổi trên dữ liệu xác thực/kiểm thử.
        """
        temp_df = X.copy()
        temp_df['target'] = y

        # Learn global statistics for each aggregation
        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        # Learn category-specific mappings
        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping

        return self

    def transform(self, X):
        """
        Áp dụng các ánh xạ đã học vào dữ liệu.
        Các danh mục chưa được nhìn thấy được điền bằng số liệu thống kê toàn cầu.
        """
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
                X_transformed[new_col_name] = X[col].map(map_series)
                X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

    def fit_transform(self, X, y):
        """
        Điều chỉnh và biến đổi dữ liệu bằng cách sử dụng phương pháp kiểm định chéo nội bộ để ngăn ngừa rò rỉ dữ liệu.
        """
        # Đầu tiên, áp dụng phương pháp khớp dữ liệu lên toàn bộ tập dữ liệu để có được các ánh xạ toàn cục cho phương pháp biến đổi
        self.fit(X, y)

        # Khởi tạo một DataFrame trống để lưu trữ các đặc trưng đã mã hóa
        encoded_features = pd.DataFrame(index=X.index)

        kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_val = X.iloc[val_idx]

            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                # --- Chỉ tính toán ánh xạ trên phần huấn luyện của fold ---
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'

                    # Tính toán số liệu thống kê toàn cầu cho phần này
                    fold_global_stat = y_train.agg(agg_func)

                    # Tính toán số liệu thống kê danh mục cho phần này
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    # --- Chỉ áp dụng làm mịn cho hàm tổng hợp 'mean' ---
                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()

                        m = self.smooth
                        if self.smooth == 'auto':
                            # Phương pháp ước lượng Bayes thực nghiệm
                            variance_between = mapping.var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].var().mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between
                            else:
                                m = 0  # Không làm mịn nếu không có biến đổi giữa các nhóm

                        # Áp dụng công thức làm mịn
                        smoothed_mapping = (counts * mapping + m * fold_global_stat) / (counts + m)
                        encoded_values = X_val[col].map(smoothed_mapping)
                    else:
                        encoded_values = X_val[col].map(mapping)

                    # Lưu trữ các giá trưng đã mã hóa cho phần này
                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stat)

        # Gộp với DataFrame gốc
        X_transformed = X.copy()
        for col in encoded_features.columns:
            X_transformed[col] = encoded_features[col]

        if self.drop_original:
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)

        return X_transformed

In [ ]:
class LibFFMEncoder(object):
    def __init__(self):
        self.encoder = 1
        self.encoding = {}
    
    def encode_for_libffm(self, row):
        # Sử dụng list concatenation thay vì concatenation chuỗi lặp lại
        parts = [str(row[0])]
        
        for i, r in enumerate(row[1:]):
            key = (i, r)
            if key not in self.encoding:
                self.encoding[key] = self.encoder
                self.encoder += 1
            parts.append(f'{i}:{self.encoding[key]}:1')
        
        return ' '.join(parts)

In [10]:
oof = np.zeros(len(train))
pred = np.zeros(len(test))

kf = KFold(n_splits=5, random_state=42, shuffle=True)

for idx, (train_idx, val_idx) in enumerate(kf.split(np.zeros(len(train)))):
    X_train, X_val = train.loc[train_idx, FEATURES], train.loc[val_idx, FEATURES]
    y_train, y_val = train.loc[train_idx, TARGET], train.loc[val_idx, TARGET]
    X_test = test.copy().fillna(0)
    y_test = X_test[TARGET]
    X_test = X_test.drop(TARGET, axis=1)

    encoder = TargetEncoder_(cols_to_encode=TE_columns, cv=5, smooth='auto', aggs=['mean'], drop_original=True)
    X_train = encoder.fit_transform(X_train, y_train)
    X_val = encoder.transform(X_val)
    X_test = encoder.transform(X_test)
    
    combined = pd.concat([X_train, X_val, X_test])
    cols = [f'TE_{c}_mean' for c in TE_columns]
    
    for c in cols:
        combined[c] = pd.cut(combined[c], bins=10)
        combined[c], _ = combined[c].factorize()
    X_train = combined[:len(X_train)]
    X_val = combined[len(X_train):len(X_train) + len(X_val)]
    X_test = combined[len(X_train) + len(X_val):]

    encoder = LibFFMEncoder()
    df_all = pd.concat([
        pd.concat([y_train, X_train], axis=1),
        pd.concat([y_val, X_val], axis=1),
        pd.concat([y_test, X_test], axis=1),
    ], axis=0, ignore_index=True)
    _ = df_all.progress_apply(lambda row: encoder.encode_for_libffm(row), axis=1)
    libffm_format_train = pd.concat([y_train, X_train], axis=1).progress_apply(
        lambda row: encoder.encode_for_libffm(row), raw=True, axis=1
    )
    libffm_format_val = pd.concat([y_val, X_val], axis=1).progress_apply(
        lambda row: encoder.encode_for_libffm(row), raw=True, axis=1
    )
    libffm_format_test = pd.concat([y_test, X_test], axis=1).progress_apply(
        lambda row: encoder.encode_for_libffm(row), raw=True, axis=1
    )

    libffm_format_train.to_csv(f'libffm_train_fold_{idx + 1}.txt', index=False, header=False)
    libffm_format_val.to_csv(f'libffm_val_fold_{idx + 1}.txt', index=False, header=False)
    libffm_format_test.to_csv(f'libffm_test_fold_{idx + 1}.txt', index=False, header=False)
    
    ffm = xl.create_ffm()
    ffm.setTrain(f'libffm_train_fold_{idx + 1}.txt')
    ffm.setValidate(f'libffm_val_fold_{idx + 1}.txt')
    ffm.fit({"task": "binary", "metric": "auc", 'nthread': 2, 'epoch': 50, 'lr': 0.05, 'lambda': 0.0001}, "model.out")
    ffm.setTest(f'libffm_val_fold_{idx + 1}.txt')
    ffm.predict("./model.out", "./output.txt")
    oof[val_idx] = pd.read_csv("./output.txt", header=None).values.flatten()
    ffm.setTest(f'libffm_test_fold_{idx + 1}.txt')
    ffm.predict("./model.out", "./output.txt")
    pred += pd.read_csv("./output.txt", header=None).values.flatten()
    print(f'Fold {idx + 1}: {np.sqrt(roc_auc_score(y_val, oof[val_idx]))}')

    del X_train, X_val, y_train, y_val
    gc.collect()

pred /= 5
print(f'CV ROC AUC: {np.sqrt(roc_auc_score(train[TARGET], oof))}')

Progress:  81%|█████████████████▊    | 384147/475355 [00:36<00:08, 10598.42it/s]

In [ ]:
submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')
submission[TARGET] = pred
submission.to_csv('xlearn-ffm_1.csv', index=False)
pd.DataFrame({'xlearn-ffm_1_oof': oof}).to_csv('xlearn-ffm_1_oof.csv', index=False)